## Ferramenta Browser do AgentCore com Extensões de Navegador

Neste exemplo, você aprenderá como usar [extensões de navegador](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-extensions.html) dentro do AgentCore Browser. 

As Extensões de Navegador permitem instalar extensões personalizadas em sessões de navegador no momento da criação da sessão. Isso permite personalizar o comportamento do navegador com suas próprias extensões para tarefas de automação, web scraping, testes e muito mais.

In [ ]:
!pip install -qU -r requirements.txt

Declarando variáveis globais

In [ ]:
import boto3
import json
import sys
from botocore.exceptions import ClientError

sys.path.append('../helpers/')

iam_boto3 = boto3.client('iam')
s3 = boto3.client('s3')
browser_boto3 = boto3.client('bedrock-agentcore-control')
browser_cli = boto3.client('bedrock-agentcore')

session = boto3.Session()
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']
REGION = session.region_name

BROWSER_NAME = "browser_with_extensions"
BUCKET_NAME = f"ac-browser-demos-{ACCOUNT_ID}-{REGION}"
AC_ROLE_NAME = "ac-browser-ext-execution-role"

### 1. Teste local com Playwright

Você pode verificar que na pasta extension, temos uma extensão pré-fabricada. 

Nesta etapa, vamos lançar uma sessão local para testar se a extensão está funcionando com o Playwright.

Assim que o navegador for iniciado, clique na extensão no chrome para vê-la funcionando localmente:

![local_extension.png](img/local_extension.png)

In [ ]:
from playwright.async_api import async_playwright

extension_path = "./extension"

async with async_playwright() as p:
    context = await p.chromium.launch_persistent_context(
        user_data_dir="./user-data",
        headless=False,
        args=[
            f"--disable-extensions-except={extension_path}",
            f"--load-extension={extension_path}"
        ]
    )
    
    page = await context.new_page()
    await page.goto("chrome://extensions/")
    await page.wait_for_timeout(2000)
    
    input("Pressione Enter para fechar...")
    await context.close()

#### 1.1 Criar um Bucket S3

Você precisa criar um Bucket S3, caso não exista, para armazenar gravações do navegador que baixaremos posteriormente.

In [ ]:
try:
    # verificar se o bucket existe
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} já existe")
except ClientError:
    # criar bucket
    create_params = {'Bucket': BUCKET_NAME}
    if REGION != 'us-east-1':
        create_params['CreateBucketConfiguration'] = {'LocationConstraint': REGION}
    s3.create_bucket(**create_params)
    print(f"Bucket {BUCKET_NAME} criado em {REGION}")

#### 1.2 Criar role IAM

Em seguida, você criará uma role IAM personalizada que será anexada ao AgentCore Browser:

In [ ]:
try: 
# Política de confiança
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }

    # Criar a role
    browser_role = iam_boto3.create_role(
        RoleName=AC_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy)
    )

    browser_role_arn = browser_role['Role']['Arn']

    print(f"Role ARN: {browser_role_arn}")

    # Política S3 para gravações
    ac_browser_policies = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:PutObject",
                    "s3:GetObject",
                    "s3:GetObjectVersion",
                    "s3:ListBucket",
                    "s3:ListMultipartUploadParts",
                    "s3:AbortMultipartUpload"
                ],
                "Resource": [
                    f"arn:aws:s3:::{BUCKET_NAME}",
                    f"arn:aws:s3:::{BUCKET_NAME}/*"
                ]
            }
        ]
    }

    # adicionar política inline S3
    iam_boto3.put_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyName='ac_custom_policies',
        PolicyDocument=json.dumps(ac_browser_policies)
    )

    # Anexar política gerenciada do Bedrock
    iam_boto3.attach_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyArn='arn:aws:iam::aws:policy/AmazonBedrockFullAccess'
    )

except ClientError as e:
    print(f'Exceção: {e}')
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        browser_role_arn = iam_boto3.get_role(RoleName=AC_ROLE_NAME)['Role']['Arn']
        print(f'Arn capturado: {browser_role_arn}')

Aguardar 10 segundos para garantir que a role IAM seja propagada

In [ ]:
import time

time.sleep(10)

#### 1.3 Criar o Browser personalizado do AgentCore

Vamos compactar nossa extensão e fazer upload para o Bucket S3:

In [ ]:
![ -f sample-extension.zip ] && rm sample-extension.zip
!cd extension && zip -r ../sample_extension.zip .
!cd ..

Fazer upload para o S3

In [ ]:
s3.upload_file(
    'sample_extension.zip',
    BUCKET_NAME,
    'extensions/sample_extension.zip',
    ExtraArgs={'ContentType': 'application/zip'}
)

Você está criando um navegador personalizado para este exemplo, mas o recurso funciona também para o navegador gerenciado (aws.browser.v1).

In [ ]:
created_browser = browser_boto3.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=browser_role_arn,
    networkConfiguration={
        'networkMode': 'PUBLIC'
    },
    recording={
        'enabled': True,
        's3Location': {
            'bucket': BUCKET_NAME,
            'prefix': 'browser_recordings/'
        }
    }
)

browser_id = created_browser['browserId']
print(f"Browser ID: {browser_id}")

### 2. Teste

Para iniciar nosso teste, vamos iniciar uma nova sessão de navegador:

In [ ]:
response = browser_cli.start_browser_session(
    browserIdentifier=browser_id,
    extensions=[
        {
            "location": {
                "s3": {
                    "bucket": BUCKET_NAME,
                    "prefix": "extensions/sample_extension.zip"
                }
            }
        }
    ]
)

session_id = response['sessionId']
print(f"Session ID: {session_id}")

Na célula seguinte, estamos assinando nossa requisição com Sigv4, para adicionar credenciais IAM nela.

In [ ]:
import browser_helper as helper

url = helper.get_url(browser_id, session_id)
headers = helper.get_signed_headers(url)
headers

#### 2.1 Testando no AgentCore Browser

Agora, vamos usar o playwright para verificar e testar nossa extensão.
[Playwright](https://playwright.dev/docs/intro) é um framework para Testes e Automação Web que é suportado pelo AgentCore Browser.

Antes de executar o código do playwright, vá até seu Browser no console AWS e clique no botão *view live session*:

![browser_console.png](img/browser_console.png)

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()
    
    
    await page.goto("chrome://extensions/")
    await page.wait_for_timeout(2000)

Após seu código ser executado, você pode abrir o navegador e clicar na extensão, para ver que está funcionando no AgentCore Browser:

![remote_extension.png](img/remote_extension.png)

#### 2.3 Encerrar sessão

Finalmente, vamos encerrar nossa sessão. 

In [ ]:
stoped_session = browser_cli.stop_browser_session(
    browserIdentifier=browser_id,
    sessionId=session_id
)
stoped_session

### 3. Limpeza (Opcional)

Excluir o navegador AgentCore personalizado e o Perfil

In [ ]:
browser_boto3.delete_browser(browserId=browser_id)